# Tutorial 6 — Dois corpos e refinamentos locais

## 1. Importações

In [ ]:
from pathlib import Path
import sys

# Importar o pacote local
project_root = Path.cwd()
if project_root.name == "tutorials":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt

from meshlab.core import Mesh, Curve, Contour, Front, Point, Refinement
from meshlab.operations import (
    insert_contour, initialize_front, compute_normals, compute_angles, advance_front,
)
from meshlab.utils import plot_mesh
from meshlab.export import export_mesh

## 2. Parâmetros

In [ ]:
num_layers = 4
height = 0.025
growth_rate = 1.15
mesh_size = 0.3
thickness = 0.1
show_gmsh = False             # True abre a malha final na janela do Gmsh

filename = project_root / "tutorials/results/t6-refinements.msh"

## 3. Círculo

In [ ]:
center = np.array([-1.8, 0.0])
radius = 0.8
num_circle_points = 64

# Sentido anti-horário
angles = np.linspace(0.0, 2.0 * np.pi, num_circle_points, endpoint=False)
circle_points = center + radius * np.column_stack((np.cos(angles), np.sin(angles)))
# Fechar a curva
circle_points = np.vstack((circle_points, circle_points[0]))

circle = Contour("circle_body", [Curve("circle", circle_points)])

## 4. Quadrado

In [ ]:
# Vértices em sentido anti-horário
corners = np.array([
    [0.8, -0.8],
    [2.4, -0.8],
    [2.4,  0.8],
    [0.8,  0.8],
])
points_per_side = 12

square_points = np.vstack([
    np.linspace(start, end, points_per_side, endpoint=False)
    for start, end in zip(corners, np.roll(corners, -1, axis=0))
])
square_points = np.vstack((square_points, square_points[0]))

square = Contour("square_body", [Curve("square", square_points)])

## 5. Montagem e camadas

In [ ]:
mesh = Mesh(dim=2)
insert_contour(mesh, circle)
insert_contour(mesh, square)

circle_front = Front("circle_front")
square_front = Front("square_front")
initialize_front(mesh, circle_front, curves=["circle"])
initialize_front(mesh, square_front, curves=["square"])
fronts = [circle_front, square_front]

h = height
for _ in range(num_layers):
    for front in fronts:
        normals = compute_normals(mesh, front, method="bisector")
        angles = compute_angles(mesh, front)
        # Corrigir a altura nos cantos do quadrado
        heights = h / np.sin(angles / 2)
        advance_front(mesh, front, normals, heights)
    h *= growth_rate

plot_mesh(mesh);

## 6. Domínio externo

In [ ]:
xmin, xmax = -5.0, 5.0
ymin, ymax = -3.0, 3.0

# Contorno externo em sentido horário
inlet = Curve("inlet", np.linspace([xmin, ymin], [xmin, ymax], 17))
laterals_upper = Curve("laterals_upper", np.linspace([xmin, ymax], [xmax, ymax], 41))
outlet = Curve("outlet", np.linspace([xmax, ymax], [xmax, ymin], 17))
laterals_lower = Curve("laterals_lower", np.linspace([xmax, ymin], [xmin, ymin], 41))

domain = Contour("outer_domain", [inlet, laterals_upper, outlet, laterals_lower])

insert_contour(mesh, domain)

## 7. Refinamento por ponto

In [ ]:
focus = Point("focus", [-2.0, 1.6])
point_refinement = Refinement(
    geometry=focus, size=0.05, distance=0.15, transition=0.65,
)

## 8. Refinamento por curva

In [ ]:
wake = Curve("wake", [[2.8, 0.0], [4.3, 0.0]])
curve_refinement = Refinement(
    geometry=wake, size=0.06, distance=0.15, transition=0.65,
)

## 9. Refinamento por região fechada

In [ ]:
region = Contour("refined_region", [
    Curve("region_outline", [
        [-2.0, -2.3], [2.0, -2.3], [2.0, -1.7], [-2.0, -1.7], [-2.0, -2.3],
    ]),
])
region_refinement = Refinement(
    geometry=region, size=0.08, distance=0.0, transition=0.5,
)

# Referências de tamanho: não inserir como fronteiras da malha
refinements = [point_refinement, curve_refinement, region_refinement]

## 10. Geometrias de refinamento

In [ ]:
figure, ax = plot_mesh(mesh, show=False)
figure.set_size_inches(12, 6)
ax.scatter(*focus.coordinates, color="tab:red", marker="x", s=70, label="Refinamento por ponto")
ax.plot(wake.points[:, 0], wake.points[:, 1], "--", color="tab:orange", label="Refinamento por curva")
polygon = region.curves[0].points
ax.fill(polygon[:, 0], polygon[:, 1], color="tab:green", alpha=0.2, label="Refinamento por região")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
figure.tight_layout()
plt.show()

## 11. Preenchimento e exportação pelo Gmsh

In [ ]:
filename.parent.mkdir(parents=True, exist_ok=True)

export_mesh(
    mesh,
    fronts=fronts,
    filename=filename,
    mode="planar",
    thickness=thickness,
    mesh_size=mesh_size,
    refinements=refinements,
    show=show_gmsh,
)

print(filename)

## 12. Conversão para OpenFOAM

In [ ]:
# Executar no terminal, dentro de um caso OpenFOAM preparado:
# gmshToFoam /caminho/para/tutorials/results/t6-refinements.msh
# Definir frontAndBack como empty em constant/polyMesh/boundary e nos campos.
# checkMesh -allTopology -allGeometry